In [1]:
import rasterio

In [3]:
import ee
import pandas as pd
import rpy2.robjects as robjects
from datetime import datetime, timedelta
import rasterio


ee.Authenticate()

ee.Initialize(project="ee-f20170834h")

Enter verification code: 4/1AdLIrYffWB0MwOmXu2ZBrSaLFSuKEleIeST71clHSZ_XrXVYtXz8umqU2us

Successfully saved authorization token.


In [4]:
def read_rds(file_path):
    readRDS = robjects.r['readRDS']
    return readRDS(file_path)

def fractional_year_to_date(fractional_year):
    year = int(fractional_year)
    remainder = fractional_year - year
    start_of_year = datetime(year, 1, 1)
    days_in_year = 366 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 365
    days_to_add = int(remainder * days_in_year)
    target_date = start_of_year + timedelta(days=days_to_add)
    return target_date.strftime('%Y-%m-%d')

In [5]:
import numpy as np
from tqdm import tqdm

In [6]:
def get_monthly_stats(image_collection, year, month, geometry):
    start_date = f"{year}-{month:02d}-01"
    end_date = f"{year}-{month:02d}-{pd.Timestamp(start_date).days_in_month}"

    monthly_collection = image_collection.filter(ee.Filter.date(start_date, end_date)).filterBounds(geometry)

    mean_temp = monthly_collection.select(['temperature_2m', 'skin_temperature']).mean()

    total_precip = monthly_collection.select(['total_precipitation']).sum()

    combined_image = mean_temp.addBands(total_precip)

    stats = combined_image.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.sum(), sharedInputs=True),
        geometry=geometry,
        scale=11000,
        maxPixels=10
    )
    return stats.getInfo()

#glaciers = ['G010748E46850N', 'G042869E43218N','G072799E39141N']
glaciers = ['G007276E45599', 'G010748E46850N']
glaciers = ['G077749E41787N', 'G078184E41824N']
glaciers = ['G010748E46850N','G077277E43048N','G077314E43095N','G077333E32259N','G077342E32257N','G077358E32209N','G077513E32227N']
glaciers = ['G077314E43095N','G077333E32259N','G077342E32257N','G077358E32209N','G077513E32227N']
glaciers = ['G007880E45990N','G007603E46067N', 'G007663E45970N', 'G007702E46120N', 'G007761E45922N', 'G007794E45916N', 'G007807E46481N', 'G007846E45906N', 'G007858E45898N', 'G007867E46119N', 'G007882E45908N', 'G007887E45948N', 'G007908E45948N', 'G007930E46472N', 'G008006E46177N', 'G008010E46463N',  'G008087E46260N']
glaciers = ['G006819E45785N','G007026E45991N']
# done glaciers = ['G007581E46002N']


for glacier in glaciers:
    print(f"Currently processing glacier {glacier}")
    file_path = f'./output/restart/03_extract_IP/{glacier}_dates_cut.rds'
    file_path = f'C:\\Users\\shash\\OneDrive\\Desktop\\Research\\Glacier Revamp\\output\\restart\\03_extract_IP\\output\\{glacier}_dates_cut.rds'
  #reading the rds file to get timestamps from it
    dates = read_rds(file_path)
    tt = np.array(dates)
    dates = [fractional_year_to_date(year) for year in tt]

  #tif file path
    path = f'C:\\Users\\shash\\OneDrive\\Desktop\\Research\\Glacier Revamp\\data\\DigitalElevationModels\\{glacier}_NASADEM.tif'
    

    with rasterio.open(path) as dataset:
        bounds = dataset.bounds
        ulx, lry, lrx, uly = bounds.left, bounds.bottom, bounds.right, bounds.top
        crs = dataset.crs.to_string()
        bbox = [[ulx, uly], [lrx, uly], [lrx, lry], [ulx, lry], [ulx, uly]]
        res = dataset.res

        geometry = ee.Geometry.Polygon([bbox])
        print(crs)
        print(res)


          # Find the earliest and latest years
        earliest_year = min(pd.to_datetime(dates).year)
        latest_year = max(pd.to_datetime(dates).year)

        # Create a list of all months within this range
        tasks = [(year, month) for year in range(earliest_year, latest_year + 1) for month in range(1, 13)]

        # Image collection
        image_collection = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY').select(['temperature_2m', 'skin_temperature', 'total_precipitation'])

          # List to store the data before creating DataFrame
        results = []



        for task in tqdm(tasks):
            year, month = task
            #print(f"Processing {year}-{month:02d}")
            stats = get_monthly_stats(image_collection, year, month, geometry)
            results.append({
                'Year': year,
                'Month': month,
                'Temperature_2m': stats.get('temperature_2m_mean'),
                'Skin_Temperature': stats.get('skin_temperature_mean'),
                'Total_Precipitation': stats.get('total_precipitation_sum')
                })
        results_df = pd.DataFrame(results)
        filename = f"{glacier}_env.csv"

  # Sort the DataFrame by Year and Month, and save
        sorted_results_df = results_df.sort_values(by=['Year', 'Month']).reset_index(drop=True)
        sorted_results_df.to_csv(filename, index=False)
        print(f"Data saved to {filename}")


Currently processing glacier G006819E45785N
EPSG:4326
(0.0002694945852358564, 0.0002694945852358564)


100%|████████████████████████████████████████████████████████████████████████████████| 456/456 [38:26<00:00,  5.06s/it]


Data saved to G006819E45785N_env.csv
Currently processing glacier G007026E45991N
EPSG:4326
(0.0002694945852358564, 0.0002694945852358564)


100%|████████████████████████████████████████████████████████████████████████████████| 444/444 [29:38<00:00,  4.01s/it]

Data saved to G007026E45991N_env.csv


In [27]:
def process_monthly_data(image):
    date = image.date().format('YYYY-MM')
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.sum(), sharedInputs=True),
        geometry=geometry,
        scale=11000,
        maxPixels=10
    )
    # Correctly constructing the feature with all statistics and date
    feature = ee.Feature(None, {'date': date}).setMulti(stats)
    return feature


def get_multi_year_stats(image_collection, start_year, end_year, geometry):
    start_date = f"{start_year}-01-01"
    
    end_date = f"{end_year}-12-31"
    multi_year_collection = image_collection.filter(ee.Filter.date(start_date, end_date)).filterBounds(geometry)
    monthly_features = multi_year_collection.map(process_monthly_data)
    return monthly_features.getInfo()  # Returns the processed data as a list of features




#glaciers = ['G010748E46850N', 'G042869E43218N','G072799E39141N']
glaciers = ['G007276E45599', 'G010748E46850N']
glaciers = ['G077749E41787N', 'G078184E41824N']
glaciers = ['G010748E46850N','G077277E43048N','G077314E43095N','G077333E32259N','G077342E32257N','G077358E32209N','G077513E32227N']
glaciers = ['G077314E43095N','G077333E32259N','G077342E32257N','G077358E32209N','G077513E32227N']
glaciers = ['G007603E46067N', 'G007663E45970N', 'G007702E46120N', 'G007761E45922N', 'G007794E45916N', 'G007807E46481N', 'G007846E45906N', 'G007858E45898N', 'G007867E46119N', 'G007880E45990N', 'G007882E45908N', 'G007887E45948N', 'G007908E45948N', 'G007930E46472N', 'G008006E46177N', 'G008010E46463N',  'G008087E46260N']

# done glaciers = ['G007581E46002N']


for glacier in glaciers:
    print(f"Currently processing glacier {glacier}")
    file_path = f'./output/restart/03_extract_IP/{glacier}_dates_cut.rds'
    file_path = f'C:\\Users\\shash\\OneDrive\\Desktop\\Research\\Glacier Revamp\\output\\restart\\03_extract_IP\\output\\{glacier}_dates_cut.rds'
  #reading the rds file to get timestamps from it
    dates = read_rds(file_path)
    tt = np.array(dates)
    dates = [fractional_year_to_date(year) for year in tt]

  #tif file path
    path = f'C:\\Users\\shash\\OneDrive\\Desktop\\Research\\Glacier Revamp\\data\\DigitalElevationModels\\{glacier}_NASADEM.tif'
    

    with rasterio.open(path) as dataset:
        bounds = dataset.bounds
        ulx, lry, lrx, uly = bounds.left, bounds.bottom, bounds.right, bounds.top
        crs = dataset.crs.to_string()
        bbox = [[ulx, uly], [lrx, uly], [lrx, lry], [ulx, lry], [ulx, uly]]
        res = dataset.res

        geometry = ee.Geometry.Polygon([bbox])
        print(crs)
        print(res)


        for year_block in range(earliest_year, latest_year + 1, 5):
                end_year = min(year_block + 4, latest_year)  # Ensure we do not exceed the latest year
                print(f"Processing from {year_block} to {end_year}")

                # Fetching stats for the current 5-year block
                stats = get_multi_year_stats(image_collection, year_block, end_year, geometry)

                # Assuming a function to convert stats to a DataFrame and save it
                # Convert the list of dictionaries into a DataFrame
                data = [{
                    'Date': feature['properties']['date'],
                    'Temperature_2m': feature['properties'].get('temperature_2m_mean', None),
                    'Skin_Temperature': feature['properties'].get('skin_temperature_mean', None),
                    'Total_Precipitation': feature['properties'].get('total_precipitation_sum', None)
                } for feature in stats['features']]

                results_df = pd.DataFrame(data)
        
        # Define the filename to include the range of years processed
        filename = f"./output/{glacier}_environmental_data_{year_block}_to_{end_year}.csv"
        results_df.to_csv(filename, index=False)
        print(f"Data saved to {filename}")


Currently processing glacier G007603E46067N
EPSG:4326
(0.0002694945852358564, 0.0002694945852358564)
Processing from 1984 to 1988


EEException: Collection query aborted after accumulating over 5000 elements.

In [28]:
def process_monthly_data(image):
    date = image.date().format('YYYY-MM')
    stats = image.reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.sum(), sharedInputs=True),
        geometry=geometry,
        scale=11000,
        maxPixels=10
    )
    # Correctly constructing the feature with all statistics and date
    feature = ee.Feature(None, {'date': date}).setMulti(stats)
    return feature

def get_multi_year_stats(image_collection, year, geometry):
    start_date = f"{year}-01-01"
    end_date = f"{year}-12-31"
    yearly_collection = image_collection.filter(ee.Filter.date(start_date, end_date)).filterBounds(geometry)
    monthly_features = yearly_collection.map(process_monthly_data)
    return monthly_features.getInfo()  # Returns the processed data as a list of features

# Loop to process each glacier, one year at a time
for glacier in glaciers:
    print(f"Currently processing glacier {glacier}")
    file_path = f'C:\\Users\\shash\\OneDrive\\Desktop\\Research\\Glacier Revamp\\output\\restart\\03_extract_IP\\output\\{glacier}_dates_cut.rds'
    dates = read_rds(file_path)
    tt = np.array(dates)
    dates = [fractional_year_to_date(year) for year in tt]

    path = f'C:\\Users\\shash\\OneDrive\\Desktop\\Research\\Glacier Revamp\\data\\DigitalElevationModels\\{glacier}_NASADEM.tif'
    with rasterio.open(path) as dataset:
        bounds = dataset.bounds
        ulx, lry, lrx, uly = bounds.left, bounds.bottom, bounds.right, bounds.top
        bbox = [[ulx, uly], [lrx, uly], [lrx, lry], [ulx, lry], [ulx, uly]]
        geometry = ee.Geometry.Polygon([bbox])
        print(f"CRS: {dataset.crs.to_string()}, Resolution: {dataset.res}")

        # Find the earliest and latest years
        earliest_year = min(pd.to_datetime(dates).year)
        latest_year = max(pd.to_datetime(dates).year)

        for year in range(earliest_year, latest_year + 1):
            print(f"Processing year {year}")
            stats = get_multi_year_stats(image_collection, year, geometry)

            # Convert the list of dictionaries into a DataFrame
            data = [{
                'Date': feature['properties']['date'],
                'Temperature_2m': feature['properties'].get('temperature_2m_mean', None),
                'Skin_Temperature': feature['properties'].get('skin_temperature_mean', None),
                'Total_Precipitation': feature['properties'].get('total_precipitation_sum', None)
            } for feature in stats['features']]

            results_df = pd.DataFrame(data)
            filename = f"./output/{glacier}_environmental_data_{year}.csv"
            results_df.to_csv(filename, index=False)
            print(f"Data saved to {filename}")


Currently processing glacier G007603E46067N
CRS: EPSG:4326, Resolution: (0.0002694945852358564, 0.0002694945852358564)
Processing year 1984


EEException: Collection query aborted after accumulating over 5000 elements.